# VAD SET-AE Training with Compressed D-Vectors

This notebook provides an interactive version of the `vad_set_ae.py` training script.

## Workflow:
1. Extract frame-level d-vectors from fbanks (streaming encoder)
2. Compress frame d-vectors: 256-dim → 64-dim (autoencoder)
3. Compress enrolled d-vector: 256-dim → 64-dim (autoencoder)
4. Compute cosine similarity between compressed d-vectors
5. Input: [40 fbanks + 64 compressed frame d-vec + 1 similarity score] = 105-dim

**Input size reduced:** 297-dim → 105-dim (2.8x smaller)

## 1. Imports and Setup

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import kaldiio

from sklearn.metrics import average_precision_score
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

import numpy as np
import os
import sys
from pathlib import Path
import pickle

# Add paths for imports
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(os.getcwd(), 'AE_test'))

from personal_vad import PersonalVAD, WPL
from resemblyzer_mod import VoiceEncoderMod
from resemblyzer import VoiceEncoder
from train_dvector_autoencoder import DvectorAutoencoder

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

import os

# Change to the src directory where the notebook is located
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir("C:\Work\Coding\Diarization\PersonalVAD\personalVAD")

print(f"Current working directory: {os.getcwd()}")

✓ Imports successful
PyTorch version: 1.13.1+cu117
CUDA available: True
Current working directory: C:\Work\Coding\Diarization\PersonalVAD\personalVAD


## 2. Configuration Parameters

In [2]:
# Training hyperparameters
num_epochs = 10
batch_size = 64
batch_size_test = 64
lr = 1e-3
SCHEDULER = True
USE_WPL = False
NUM_WORKERS = 0

# Autoencoder configuration
USE_AUTOENCODER = False  # Set to False to use full 256-dim d-vectors instead of compressed 64-dim

# Model architecture (will be set based on USE_AUTOENCODER)
if USE_AUTOENCODER:
    input_dim = 105  # 40 fbanks + 64 compressed d-vector + 1 score
    dvector_dim = 64
else:
    input_dim = 297  # 40 fbanks + 256 full d-vector + 1 score
    dvector_dim = 256

hidden_dim = 64
out_dim = 3
num_layers = 2

# Paths - Updated to use correct paths
DATA_TRAIN = 'data/overlap_50pct_babble_main84_500'
DATA_TEST = 'data/overlap_50pct_babble_main84_50'
EMBED_PATH = 'data/embeddings'
AE_MODEL_PATH = 'src/AE_test/test_outputs/dvector_autoencoder_non_target_aware_zero'
MODEL_PATH = f'vad_set{"_ae" if USE_AUTOENCODER else ""}.pt'

# Visualization options
SAVE_COMPRESSED_DVECTORS = True
COMPRESSED_OUTPUT_DIR = 'compressed_dvectors_output'

# Other
SCORE_TYPE = 0
SAVE_MODEL = True

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
WPL_WEIGHTS = torch.tensor([1.0, 0.1, 1.0]).to(device)

print("Configuration:")
print(f"  Device: {device}")
print(f"  USE AUTOENCODER: {USE_AUTOENCODER}")
print(f"  D-vector dimension: {dvector_dim}")
print(f"  Epochs: {num_epochs}")
print(f"  Batch size: {batch_size}")
print(f"  Learning rate: {lr}")
print(f"  Input dim: {input_dim}")
print(f"  Hidden dim: {hidden_dim}")
print(f"  Output classes: {out_dim}")
print(f"\nPaths:")
print(f"  Train data: {DATA_TRAIN}")
print(f"  Test data: {DATA_TEST}")
print(f"  Embeddings: {EMBED_PATH}")
if USE_AUTOENCODER:
    print(f"  Autoencoder: {AE_MODEL_PATH}")
print(f"  Model save path: {MODEL_PATH}")

Configuration:
  Device: cuda
  USE AUTOENCODER: False
  D-vector dimension: 256
  Epochs: 10
  Batch size: 64
  Learning rate: 0.001
  Input dim: 297
  Hidden dim: 64
  Output classes: 3

Paths:
  Train data: data/overlap_50pct_babble_main84_500
  Test data: data/overlap_50pct_babble_main84_50
  Embeddings: data/embeddings
  Model save path: vad_set.pt


## 3. Helper Functions

In [3]:
def load_autoencoder(model_dir, device='cuda'):
    """Load trained autoencoder model"""
    model_dir = Path(model_dir)
    
    # If path is relative, resolve it from script location
    if not model_dir.is_absolute():
        model_dir = Path(os.getcwd()) / model_dir
    
    # Load config
    config_path = model_dir / 'config.pkl'
    if not config_path.exists():
        raise FileNotFoundError(f"Config not found: {config_path}")
    
    with open(config_path, 'rb') as f:
        config = pickle.load(f)
    
    # Create model
    autoencoder = DvectorAutoencoder(
        input_dim=config['input_dim'],
        hidden_dims=config['hidden_dims'],
        dropout_rate=config['dropout_rate']
    )
    
    # Load weights
    model_path = model_dir / 'final_model.pth'
    if not model_path.exists():
        model_path = model_dir / 'best_model.pth'
        if not model_path.exists():
            raise FileNotFoundError(f"Model not found in: {model_dir}")
    
    if device == 'cuda' and torch.cuda.is_available():
        autoencoder.load_state_dict(torch.load(model_path))
        autoencoder = autoencoder.to(device)
    else:
        autoencoder.load_state_dict(torch.load(model_path, map_location='cpu'))
        autoencoder = autoencoder.to('cpu')
    
    autoencoder.eval()
    
    print(f"✓ Loaded autoencoder from: {model_dir}")
    print(f"  Architecture: {config['hidden_dims']}")
    print(f"  Bottleneck dimension: {config['hidden_dims'][len(config['hidden_dims'])//2]}")
    
    return autoencoder, config


def pad_collate_with_metadata(batch):
    """Custom padding function that handles additional metadata from dataset"""
    (xx, yy, keys, compressed_frames, original_frames, enrolled_compressed, overlap_times) = zip(*batch)
    x_lens = [len(x) for x in xx]
    y_lens = [len(y) for y in yy]

    x_padded = pad_sequence(xx, batch_first=True, padding_value=0)
    y_padded = pad_sequence(yy, batch_first=True, padding_value=0)

    return x_padded, y_padded, x_lens, y_lens, keys, compressed_frames, original_frames, enrolled_compressed, overlap_times

print("✓ Helper functions defined")

✓ Helper functions defined


## 4. t-SNE Visualization Function

In [4]:
def plot_tsne_compressed_dvectors(compressed_frame, original_frame, enrolled_compressed, 
                                   enrolled_original, labels, utt_key, output_dir, overlap_regions=None):
    """Create t-SNE visualization comparing original and compressed d-vectors for one utterance
    
    Args:
        overlap_regions: List of (start_frame, end_frame) tuples indicating overlapping speech regions
    """
    
    # Combine frame and enrolled d-vectors
    original_all = np.vstack([original_frame, enrolled_original.reshape(1, -1)])
    compressed_all = np.vstack([compressed_frame, enrolled_compressed.reshape(1, -1)])
    
    n_frames = len(compressed_frame)
    
    # Map frame labels: 0=non-speech, 1=non-target, 2=target
    label_names = ['non-speech', 'non-target', 'target']
    frame_colors = []
    
    # Create overlap mask
    is_overlap = np.zeros(n_frames, dtype=bool)
    if overlap_regions is not None:
        for start, end in overlap_regions:
            is_overlap[start:end] = True
    
    for i, l in enumerate(labels):
        base_label = label_names[int(l)]
        if is_overlap[i] and base_label == 'target':
            frame_colors.append('target-overlap')
        elif is_overlap[i] and base_label == 'non-target':
            frame_colors.append('non-target-overlap')
        else:
            frame_colors.append(base_label)
    
    frame_colors.append('enrolled')
    
    # t-SNE for original and compressed
    print(f"  Running t-SNE on original 256-dim d-vectors...")
    tsne_original = TSNE(n_components=2, random_state=42, perplexity=min(30, n_frames-1))
    embedded_original = tsne_original.fit_transform(original_all)
    
    print(f"  Running t-SNE on compressed 64-dim d-vectors...")
    tsne_compressed = TSNE(n_components=2, random_state=42, perplexity=min(30, n_frames-1))
    embedded_compressed = tsne_compressed.fit_transform(compressed_all)
    
    # Create side-by-side plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Color mapping
    color_map = {
        'non-speech': 'gray',
        'target': 'green',
        'non-target': 'red',
        'target-overlap': 'darkgreen',
        'non-target-overlap': 'darkred',
        'enrolled': 'blue'
    }
    
    # Plot both graphs
    for i, label in enumerate(frame_colors):
        color = color_map[label]
        marker = 'o' if i < n_frames else '*'
        size = 50 if i < n_frames else 300
        ax1.scatter(embedded_original[i, 0], embedded_original[i, 1], 
                   c=color, marker=marker, s=size, alpha=0.6)
        ax2.scatter(embedded_compressed[i, 0], embedded_compressed[i, 1], 
                   c=color, marker=marker, s=size, alpha=0.6)
    
    ax1.set_title(f'Original 256-dim d-vectors\n{utt_key}', fontsize=12)
    ax1.set_xlabel('t-SNE dimension 1')
    ax1.set_ylabel('t-SNE dimension 2')
    ax1.grid(True, alpha=0.3)
    
    ax2.set_title(f'Compressed 64-dim d-vectors\n{utt_key}', fontsize=12)
    ax2.set_xlabel('t-SNE dimension 1')
    ax2.set_ylabel('t-SNE dimension 2')
    ax2.grid(True, alpha=0.3)
    
    # Add legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='green', markersize=8, 
               label='Target speaker (clean)', alpha=0.6),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='darkgreen', markersize=8, 
               label='Target speaker (overlap)', alpha=0.6),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=8, 
               label='Non-target speaker', alpha=0.6),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='darkred', markersize=8, 
               label='Non-target (overlap)', alpha=0.6),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=8, 
               label='Non-speech/Silence', alpha=0.6),
        Line2D([0], [0], marker='*', color='w', markerfacecolor='blue', markersize=12, 
               label='Enrolled speaker', alpha=0.6),
    ]
    
    ax1.legend(handles=legend_elements, loc='best', fontsize=9)
    ax2.legend(handles=legend_elements, loc='best', fontsize=9)
    
    plt.tight_layout()
    
    # Save plot
    output_path = Path(output_dir) / f'tsne_{utt_key}.png'
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"  ✓ Saved t-SNE plot: {output_path}")
    
    return output_path

print("✓ Visualization function defined")

✓ Visualization function defined


## 5. Dataset Class

In [5]:
class VadSETAEDataset(Dataset):
    """VadSET dataset with optional autoencoder-compressed d-vectors and recomputed scores.
    
    Note: All models (autoencoder, encoder_stream) are kept on CPU during data loading
    to avoid CUDA multiprocessing issues with DataLoader workers.
    """

    def __init__(self, root_dir, embed_path, score_type, encoder_stream, autoencoder=None, use_autoencoder=True):
        self.root_dir = root_dir
        self.embed_path = embed_path
        self.score_type = score_type
        self.use_autoencoder = use_autoencoder
        
        # Keep models on CPU for DataLoader compatibility
        self.encoder_stream = encoder_stream.cpu()
        if autoencoder is not None:
            self.autoencoder = autoencoder.cpu()
        else:
            self.autoencoder = None

        # load the scp files...
        self.fbanks = kaldiio.load_scp(f'{self.root_dir}/fbanks.scp')
        self.labels = kaldiio.load_scp(f'{self.root_dir}/labels.scp')
        self.keys = np.array(list(self.fbanks))
        self.embed = kaldiio.load_scp(f'{self.embed_path}/dvectors.scp')

        # load the target speaker ids
        self.targets = {}
        with open(f'{self.root_dir}/targets.scp') as targets:
            for line in targets:
                (utt_id, target) = line.split()
                self.targets[utt_id] = target
        
        # Try to load overlap metadata if available
        self.overlap_metadata = self._load_overlap_metadata()
        
        # Pre-compress enrolled d-vectors if using autoencoder
        if self.use_autoencoder and self.autoencoder is not None:
            print(f"Compressing enrolled d-vectors with autoencoder: 256-dim → 64-dim...")
            self.compressed_embed = {}
            self.autoencoder.eval()
            with torch.no_grad():
                for target, dvector in self.embed.items():
                    dvector_tensor = torch.FloatTensor(dvector).unsqueeze(0)
                    compressed = self.autoencoder.encode(dvector_tensor)
                    self.compressed_embed[target] = compressed.cpu().numpy().squeeze()
            print(f"✓ Compressed {len(self.compressed_embed)} enrolled d-vectors")
        else:
            print(f"Using full 256-dim d-vectors (no compression)")
            self.compressed_embed = None
    
    def _load_overlap_metadata(self):
        """Load overlap region information from .overlap_meta files if they exist"""
        overlap_data = {}
        
        wav_scp_path = f'{self.root_dir}/wav.scp'
        if not os.path.exists(wav_scp_path):
            return overlap_data
        
        with open(wav_scp_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 2:
                    continue
                utt_id = parts[0]
                
                if '.flac' in line:
                    flac_path = line.split('.flac')[0].split()[-1] + '.flac'
                    meta_path = flac_path.replace('.flac', '.overlap_meta')
                    
                    if os.path.exists(meta_path):
                        overlap_data[utt_id] = self._parse_overlap_meta(meta_path)
        
        if overlap_data:
            print(f"✓ Loaded overlap metadata for {len(overlap_data)} utterances")
        
        return overlap_data
    
    def _parse_overlap_meta(self, meta_path):
        """Parse .overlap_meta file to extract overlap time ranges"""
        overlaps = []
        with open(meta_path, 'r') as f:
            for line in f:
                line = line.strip()
                if line.startswith('overlap_'):
                    parts = line.split()
                    if len(parts) >= 2:
                        time_range = parts[1]
                        if '-' in time_range:
                            start, end = time_range.split('-')
                            overlaps.append((float(start), float(end)))
        return overlaps

    def __len__(self):
        return self.keys.size

    def __getitem__(self, idx):
        key = self.keys[idx]
        target = self.targets[key]
        fbanks = self.fbanks[key]  # (n_frames, 40)
        y = self.labels[key]

        # 1. Extract frame-level d-vectors from fbanks using streaming encoder (CPU)
        with torch.no_grad():
            fbanks_tensor = torch.FloatTensor(fbanks).unsqueeze(0)  # (1, n_frames, 40)
            frame_dvectors, _ = self.encoder_stream.forward_stream(fbanks_tensor, None)
            frame_dvectors = frame_dvectors.squeeze(0).numpy()  # (n_frames, 256)
        
        if self.use_autoencoder and self.autoencoder is not None:
            # 2a. Compress frame-level d-vectors with autoencoder (CPU)
            with torch.no_grad():
                frame_tensor = torch.FloatTensor(frame_dvectors)  # (n_frames, 256)
                compressed = self.autoencoder.encode(frame_tensor)  # (n_frames, 64)
                compressed_frame_dvectors = compressed.numpy()  # (n_frames, 64)
            
            # Get compressed enrolled d-vector
            enrolled_compressed = self.compressed_embed[target]  # (64,)
            
            # 3a. Compute cosine similarity between compressed enrolled and compressed frame d-vectors
            enrolled_norm = enrolled_compressed / (np.linalg.norm(enrolled_compressed) + 1e-8)
            frame_norms = compressed_frame_dvectors / (np.linalg.norm(compressed_frame_dvectors, axis=1, keepdims=True) + 1e-8)
            scores = np.dot(frame_norms, enrolled_norm)  # (n_frames,)
            
            # 4a. Concatenate features: [40 fbanks + 64 compressed frame d-vector + 1 score] = 105-dim
            x = np.hstack((fbanks, compressed_frame_dvectors))  # (n_frames, 104)
            x = np.hstack((x, np.expand_dims(scores, 1)))  # (n_frames, 105)
            
            enrolled_dvector = enrolled_compressed
        else:
            # 2b. Use full 256-dim d-vectors (no compression)
            full_frame_dvectors = frame_dvectors  # (n_frames, 256)
            compressed_frame_dvectors = None
            
            # Get full enrolled d-vector
            enrolled_full = self.embed[target]  # (256,)
            
            # 3b. Compute cosine similarity between full enrolled and full frame d-vectors
            enrolled_norm = enrolled_full / (np.linalg.norm(enrolled_full) + 1e-8)
            frame_norms = full_frame_dvectors / (np.linalg.norm(full_frame_dvectors, axis=1, keepdims=True) + 1e-8)
            scores = np.dot(frame_norms, enrolled_norm)  # (n_frames,)
            
            # 4b. Concatenate features: [40 fbanks + 256 full frame d-vector + 1 score] = 297-dim
            x = np.hstack((fbanks, full_frame_dvectors))  # (n_frames, 296)
            x = np.hstack((x, np.expand_dims(scores, 1)))  # (n_frames, 297)
            
            enrolled_dvector = enrolled_full

        x = torch.from_numpy(x.copy()).float()
        y = torch.from_numpy(y.copy()).long()
        
        # Get overlap regions for this utterance (if available)
        overlap_times = self.overlap_metadata.get(key, [])
        
        # Return compressed_frame_dvectors for visualization (will be None if not using autoencoder)
        return x, y, key, compressed_frame_dvectors, frame_dvectors, enrolled_dvector, overlap_times

print("✓ Dataset class defined")

✓ Dataset class defined


## 6. Load Models and Data

In [6]:
print("=" * 80)
if USE_AUTOENCODER:
    print("VAD SET WITH AUTOENCODER-COMPRESSED D-VECTORS")
    print("=" * 80)
    print(f"Input dimension: {input_dim} (40 fbanks + 64 compressed frame d-vector + 1 score)")
    print(f"Original VAD SET input: 297 (40 fbanks + 256 d-vector + 1 score)")
    print(f"Compression ratio: {297/input_dim:.2f}x smaller")
    print("\nWorkflow:")
    print("  1. Extract frame-level d-vectors from fbanks (streaming encoder)")
    print("  2. Compress frame d-vectors: 256-dim → 64-dim (autoencoder)")
    print("  3. Compress enrolled d-vector: 256-dim → 64-dim (autoencoder)")
    print("  4. Compute cosine similarity between compressed d-vectors")
    print("  5. Input: [fbanks + compressed frame d-vec + similarity score]")
else:
    print("VAD SET WITH FULL D-VECTORS (NO AUTOENCODER)")
    print("=" * 80)
    print(f"Input dimension: {input_dim} (40 fbanks + 256 full frame d-vector + 1 score)")
    print("\nWorkflow:")
    print("  1. Extract frame-level d-vectors from fbanks (streaming encoder)")
    print("  2. Compute cosine similarity between enrolled and frame d-vectors")
    print("  3. Input: [fbanks + full frame d-vec + similarity score]")
print("=" * 80)

VAD SET WITH FULL D-VECTORS (NO AUTOENCODER)
Input dimension: 297 (40 fbanks + 256 full frame d-vector + 1 score)

Workflow:
  1. Extract frame-level d-vectors from fbanks (streaming encoder)
  2. Compute cosine similarity between enrolled and frame d-vectors
  3. Input: [fbanks + full frame d-vec + similarity score]


In [7]:
# Load autoencoder (only if USE_AUTOENCODER is True)
if USE_AUTOENCODER:
    print("\nLoading autoencoder...")
    autoencoder, ae_config = load_autoencoder(AE_MODEL_PATH, device)
else:
    print("\nSkipping autoencoder (using full 256-dim d-vectors)")
    autoencoder = None
    ae_config = None


Skipping autoencoder (using full 256-dim d-vectors)


In [8]:
# Load streaming encoder for frame-level d-vector extraction
print("\nLoading streaming voice encoder...")
encoder = VoiceEncoder(device='cpu')
encoder_stream = VoiceEncoderMod()
encoder_stream.load_state_dict(encoder.state_dict())
encoder_stream.eval()
print("✓ Streaming voice encoder loaded (CPU mode for DataLoader compatibility)")


Loading streaming voice encoder...
Loaded the voice encoder model on cpu in 0.02 seconds.
Loaded the voice encoder model on cuda in 3.82 seconds.
✓ Streaming voice encoder loaded (CPU mode for DataLoader compatibility)
Loaded the voice encoder model on cuda in 3.82 seconds.
✓ Streaming voice encoder loaded (CPU mode for DataLoader compatibility)


In [9]:
# Load datasets
print("\nLoading training data...")
train_data = VadSETAEDataset(DATA_TRAIN, EMBED_PATH, SCORE_TYPE, encoder_stream, autoencoder, USE_AUTOENCODER)

print("\nLoading test data...")
test_data = VadSETAEDataset(DATA_TEST, EMBED_PATH, SCORE_TYPE, encoder_stream, autoencoder, USE_AUTOENCODER)


Loading training data...
Using full 256-dim d-vectors (no compression)

Loading test data...
Using full 256-dim d-vectors (no compression)


In [10]:
# Create data loaders
train_loader = DataLoader(
    dataset=train_data, num_workers=NUM_WORKERS, pin_memory=True,
    batch_size=batch_size, shuffle=True, collate_fn=pad_collate_with_metadata)

test_loader = DataLoader(
    dataset=test_data, num_workers=NUM_WORKERS, pin_memory=True,
    batch_size=batch_size_test, shuffle=False, collate_fn=pad_collate_with_metadata)

print(f"✓ Data loaders created")
print(f"  Training batches: {len(train_loader)}")
print(f"  Test batches: {len(test_loader)}")

✓ Data loaders created
  Training batches: 2
  Test batches: 2


## 7. Initialize Model and Training Components

In [11]:
# Create model
model = PersonalVAD(input_dim, hidden_dim, num_layers, out_dim, use_fc=True, linear=False).to(device)

print(f"\n🏗️  VAD Model:")
if USE_AUTOENCODER:
    print(f"  Input: {input_dim}-dim (40 fbanks + 64 compressed frame d-vec + 1 compressed score)")
else:
    print(f"  Input: {input_dim}-dim (40 fbanks + 256 full frame d-vec + 1 score)")
print(f"  Hidden: {hidden_dim}-dim")
print(f"  Layers: {num_layers}")
print(f"  Output: {out_dim} classes")

total_params = sum(p.numel() for p in model.parameters())
print(f"  Total parameters: {total_params:,}")
if USE_AUTOENCODER:
    print(f"\n💡 Note: Scores are recomputed from compressed d-vectors during training")
else:
    print(f"\n💡 Note: Using full 256-dim d-vectors for speaker verification")


🏗️  VAD Model:
  Input: 297-dim (40 fbanks + 256 full frame d-vec + 1 score)
  Hidden: 64-dim
  Layers: 2
  Output: 3 classes
  Total parameters: 130,563

💡 Note: Using full 256-dim d-vectors for speaker verification


In [12]:
# Setup training components
if USE_WPL:
    criterion = WPL(WPL_WEIGHTS)
else:
    criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

if SCHEDULER:
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.1)

softmax = nn.Softmax(dim=1)

print("✓ Training components initialized")
print(f"  Loss: {'Weighted Pairwise Loss' if USE_WPL else 'CrossEntropyLoss'}")
print(f"  Optimizer: Adam (lr={lr})")
print(f"  Scheduler: {'StepLR' if SCHEDULER else 'None'}")

✓ Training components initialized
  Loss: CrossEntropyLoss
  Optimizer: Adam (lr=0.001)
  Scheduler: StepLR


## 8. Inspect First Batch (Optional)

In [13]:
# Get first batch for inspection
first_batch = next(iter(train_loader))
x_padded, y_padded, x_lens, y_lens, keys, compressed_frames, original_frames, enrolled_compressed, overlap_times = first_batch

print("First batch inspection:")
print(f"  Batch size: {len(keys)}")
print(f"  Input shape (padded): {x_padded.shape}")
print(f"  Labels shape (padded): {y_padded.shape}")
print(f"  Example utterance: {keys[0]}")
print(f"  Example length: {x_lens[0]} frames")

if USE_AUTOENCODER and compressed_frames[0] is not None:
    print(f"  Compressed frame d-vectors shape: {compressed_frames[0].shape}")
    print(f"  Enrolled compressed d-vector shape: {enrolled_compressed[0].shape}")
else:
    print(f"  Full frame d-vectors shape: {original_frames[0].shape}")
    print(f"  Enrolled full d-vector shape: {enrolled_compressed[0].shape}")

print(f"  Original frame d-vectors shape: {original_frames[0].shape}")

# Show label distribution
labels_first = y_padded[0][:y_lens[0]].numpy()
unique, counts = np.unique(labels_first, return_counts=True)
print(f"\n  Label distribution (first utterance):")
label_names = ['non-speech', 'non-target', 'target']
for u, c in zip(unique, counts):
    print(f"    {label_names[u]}: {c} frames ({c/len(labels_first)*100:.1f}%)")

if overlap_times[0]:
    print(f"\n  Overlap regions: {len(overlap_times[0])}")
    for i, (start, end) in enumerate(overlap_times[0][:3]):  # Show first 3
        print(f"    {i+1}. {start:.2f}s - {end:.2f}s")

c:\Users\user\anaconda3\envs\personal_vad\lib\site-packages\ipykernel_launcher.py:105: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_numpy.cpp:205.)


First batch inspection:
  Batch size: 64
  Input shape (padded): torch.Size([64, 3076, 297])
  Labels shape (padded): torch.Size([64, 3076])
  Example utterance: 84-121550-0009_84-121550-0018_84-121550-0014_OV0_NTSS0-reverb
  Example length: 3018 frames
  Full frame d-vectors shape: (3018, 256)
  Enrolled full d-vector shape: (256,)
  Original frame d-vectors shape: (3018, 256)

  Label distribution (first utterance):
    non-speech: 371 frames (12.3%)
    non-target: 400 frames (13.3%)
    target: 2247 frames (74.5%)


In [19]:
# Debug: Check actual values in the batch
print("\n=== DEBUGGING DATA VALUES ===")
print(f"x_padded sample values (first utterance, frame 100):")
print(f"  First 5 dims: {x_padded[0][100][:5]}")
print(f"  Last 5 dims: {x_padded[0][100][-5:]}")
print(f"  Min value: {x_padded[0][:x_lens[0]].min():.6f}")
print(f"  Max value: {x_padded[0][:x_lens[0]].max():.6f}")
print(f"  Mean value: {x_padded[0][:x_lens[0]].mean():.6f}")
print(f"\ny_padded sample values (first utterance):")
print(f"  First 10 labels: {y_padded[0][:10]}")
print(f"  Unique labels: {torch.unique(y_padded[0][:y_lens[0]])}")
print(f"\nChecking if data is all zeros:")
print(f"  x_padded all zeros? {(x_padded[0][:x_lens[0]] == 0).all()}")
print(f"  y_padded all zeros? {(y_padded[0][:y_lens[0]] == 0).all()}")


=== DEBUGGING DATA VALUES ===
x_padded sample values (first utterance, frame 100):
  First 5 dims: tensor([-1.7063, -1.9641, -2.3805, -1.4524, -0.7574])
  Last 5 dims: tensor([0.0000, 0.0000, 0.1039, 0.0000, 0.2695])
  Min value: -5.992775
  Max value: 1.499587
  Mean value: -0.363785

y_padded sample values (first utterance):
  First 10 labels: tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1], device='cuda:0')
  Unique labels: tensor([0, 1, 2], device='cuda:0')

Checking if data is all zeros:
  x_padded all zeros? False
  y_padded all zeros? False


## 9. Create t-SNE Visualization for First Utterance

In [14]:
# if SAVE_COMPRESSED_DVECTORS:
#     output_dir = Path(COMPRESSED_OUTPUT_DIR)
#     output_dir.mkdir(exist_ok=True)
#     print(f"📁 Output directory: {output_dir}")
    
#     # Save compressed d-vectors for all utterances in first batch
#     print(f"\n💾 Saving compressed d-vectors from first batch...")
#     for j in range(len(keys)):
#         utt_key = keys[j]
#         compressed_frame = compressed_frames[j]
#         original_frame = original_frames[j]
        
#         # Save compressed d-vectors
#         save_path = output_dir / f'{utt_key}_compressed.npy'
#         np.save(save_path, compressed_frame)
        
#         # Save original d-vectors for comparison
#         save_path_orig = output_dir / f'{utt_key}_original.npy'
#         np.save(save_path_orig, original_frame)
        
#         print(f"  ✓ Saved: {utt_key} (compressed: {compressed_frame.shape}, original: {original_frame.shape})")
    
#     print("\n📊 Creating t-SNE visualization for first utterance...")
#     # Get labels for first utterance
#     labels_first = y_padded[0][:y_lens[0]].cpu().numpy()
    
#     # Get enrolled original d-vector
#     enrolled_target = train_data.targets[keys[0]]
#     enrolled_original = train_data.embed[enrolled_target]
    
#     # Convert overlap times to frame indices (assuming 10ms frame shift)
#     overlap_regions = None
#     if overlap_times[0]:
#         overlap_regions = []
#         frame_shift = 0.01  # 10ms
#         for start_time, end_time in overlap_times[0]:
#             start_frame = int(start_time / frame_shift)
#             end_frame = int(end_time / frame_shift)
#             overlap_regions.append((start_frame, end_frame))
    
#     plot_tsne_compressed_dvectors(
#         compressed_frames[0], 
#         original_frames[0],
#         enrolled_compressed[0],
#         enrolled_original,
#         labels_first,
#         keys[0],
#         output_dir,
#         overlap_regions
#     )

## 10. Training Loop

In [20]:
print("\n" + "=" * 80)
print("TRAINING")
print("=" * 80)

for epoch in range(num_epochs):
    print(f"\n====== Starting epoch {epoch} ======")
    
    # Training
    model.train()
    for batch, (x_padded, y_padded, x_lens, y_lens, keys, _, _, _, _) in enumerate(train_loader):
        y_padded = y_padded.to(device)

        # Forward pass
        out_padded, _ = model(x_padded.to(device), x_lens, None)
        print(x_padded.shape, x_padded[0][100])
        print(y_padded.shape, y_padded[0][100])
        print(out_padded.shape, out_padded[0][100])

        # Compute loss
        loss = 0
        for j in range(out_padded.size(0)):
            loss += criterion(out_padded[j][:y_lens[j]], y_padded[j][:y_lens[j]])

        loss /= batch_size
        
        # Backward pass
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        if batch % 10 == 0:
            print(f'Batch: {batch}, loss = {loss:.4f}')

    # Learning rate scheduling
    if SCHEDULER and epoch < 2:
        scheduler.step()
        if epoch == 1:
            optimizer.param_groups[0]['lr'] = 5e-5
    if SCHEDULER and epoch == 7:
        optimizer.param_groups[0]['lr'] = 1e-5

    # Testing
    model.eval()
    with torch.no_grad():
        print("testing...")
        n_correct = 0
        n_samples = 0
        targets = []
        outputs = []
        
        for x_padded, y_padded, x_lens, y_lens, keys, _, _, _, _ in test_loader:
            y_padded = y_padded.to(device)
            out_padded, _ = model(x_padded.to(device), x_lens, None)

            for j in range(out_padded.size(0)):
                classes = torch.argmax(out_padded[j][:y_lens[j]], dim=1)
                n_samples += y_lens[j]
                n_correct += torch.sum(classes == y_padded[j][:y_lens[j]]).item()

                # Average precision
                p = softmax(out_padded[j][:y_lens[j]])
                outputs.append(p.cpu().numpy())
                targets.append(y_padded[j][:y_lens[j]].cpu().numpy())

        acc = 100.0 * n_correct / n_samples
        print(f"accuracy = {acc:.2f}")

        # Average precision
        targets = np.concatenate(targets)
        outputs = np.concatenate(outputs)
        targets_oh = np.eye(3)[targets]
        out_AP = average_precision_score(targets_oh, outputs, average=None)
        mAP = average_precision_score(targets_oh, outputs, average='micro')

        print(f"AP per class: {out_AP}")
        print(f"mAP: {mAP:.4f}")

    # Save model
    if SAVE_MODEL:
        path_seg = MODEL_PATH.split('/')[:-1]
        if path_seg != []:
            os.makedirs('/'.join(path_seg), exist_ok=True)
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"Model saved to {MODEL_PATH}")

print("\n" + "=" * 80)
print("✅ TRAINING COMPLETE")
print("=" * 80)


TRAINING

====== Starting epoch 0 ======
torch.Size([64, 3076, 297]) tensor([-3.0448e+00, -5.6334e-01, -4.2240e-01, -3.0581e-01,  1.0559e+00,
         9.6330e-01, -7.7890e-01, -3.3286e-01, -1.1805e+00, -1.8207e+00,
        -1.9409e+00, -1.9613e+00, -1.1420e+00, -1.7314e+00, -1.8405e+00,
        -1.6655e+00, -1.5073e+00, -1.2604e+00, -1.1267e+00, -8.9086e-01,
        -7.2501e-01, -1.4846e+00, -1.7839e+00, -1.8865e+00, -1.7803e+00,
        -1.7814e+00, -1.8445e+00, -3.0694e+00, -3.3097e+00, -3.0964e+00,
        -2.5608e+00, -2.7751e+00, -3.5116e+00, -4.0081e+00, -3.6687e+00,
        -3.7091e+00, -3.8308e+00, -4.3444e+00, -4.1302e+00, -4.3316e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  6.9178e-02,
         0.0000e+00,  1.9539e-01,  7.3000e-03,  3.6412e-02,  0.0000e+00,
         5.0215e-02,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  3.5221e-02,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  1.2735e-01,  6.5678e-02,  0.0000

KeyboardInterrupt: 

## 11. Final Summary

In [ ]:
print(f"\n💾 Model saved to: {MODEL_PATH}")
print(f"📊 Final accuracy: {acc:.2f}%")
print(f"📊 Final mAP: {mAP:.4f}")

if USE_AUTOENCODER:
    print(f"\n💡 This model uses compressed d-vectors and recomputed scores:")
    print(f"   - Frame d-vectors: 256-dim → 64-dim (autoencoder)")
    print(f"   - Enrolled d-vector: 256-dim → 64-dim (autoencoder)")
    print(f"   - Scores: Cosine similarity of compressed d-vectors")
    print(f"   - Input size reduced: 297-dim → {input_dim}-dim ({297/input_dim:.2f}x smaller)")
    
    if SAVE_COMPRESSED_DVECTORS:
        print(f"\n📁 Compressed d-vectors saved to: {output_dir}")
        print(f"📊 t-SNE visualization saved")
else:
    print(f"\n💡 This model uses full 256-dim d-vectors:")
    print(f"   - Frame d-vectors: 256-dim (no compression)")
    print(f"   - Enrolled d-vector: 256-dim (no compression)")
    print(f"   - Scores: Cosine similarity of full d-vectors")
    print(f"   - Input size: {input_dim}-dim")


💾 Model saved to: vad_set_ae.pt
📊 Final accuracy: 63.30%
📊 Final mAP: 0.6410

💡 This model uses compressed d-vectors and recomputed scores:
   - Frame d-vectors: 256-dim → 64-dim (autoencoder)
   - Enrolled d-vector: 256-dim → 64-dim (autoencoder)
   - Scores: Cosine similarity of compressed d-vectors
   - Input size reduced: 297-dim → 105-dim (2.83x smaller)

📁 Compressed d-vectors saved to: compressed_dvectors_output
📊 t-SNE visualization saved
